In [1]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/dungeon_10k_4_8_3_5_mkr.jsonl'

In [2]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.config import DataConfig
from origami.training import NotebookCallback, accuracy

config = OrigamiConfig(
    data=DataConfig(
        infer_schema=True,
        numeric_mode="disabled",
    ),
    model=ModelConfig(
        backbone="transformer",
        kvpe_pooling="sum",
        d_model=128,
        n_heads=8,
        n_layers=6,
        d_ff=784,
        dropout=0.0,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        batch_size=64,
        warmup_steps=1000,
        learning_rate=1e-3,
        eval_strategy="epoch",
        eval_epochs=1,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        eval_on_train=True,
        target_key="treasure",
        target_loss_weight=1.0,
        constrain_grammar=True,
        constrain_schema=True,
        lr_scheduler="cosine",
        lr_cosine_exponent=2.0,
    ),
    device="mps",
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data,
    eval_data=eval_data,
    callbacks=[
        NotebookCallback(
            metrics=["loss", "val_loss", "lr", "val_acc"], plot_update_interval=1, smoothing=0.5
        )
    ],
    epochs=100,
    verbose=True,
)

Numba available: False
Infer schema: True
Vocabulary size: 40
Model parameters: 1,652,360
Training device: mps


OrigamiPipeline(numeric_mode='disabled', fitted)

In [ ]:
# pipeline.save("dungeon_pipeline.pt")

In [ ]:
from origami import OrigamiPipeline

# pipeline = OrigamiPipeline.load("dungeon_pipeline.pt")

In [ ]:
from origami.training import accuracy

pipeline.evaluate(eval_data, metrics={"acc": accuracy})

In [ ]:
doc = pipeline.generate(1)[0]

print(json.dumps(doc, indent=2))